In [ ]:
####Full Lasso Logit Code ########


#Logistic Lasso Model with Full Sample 
import numpy as np
import pandas as pd

dir_data = "../"   # specify the directory to data files
dir_lasso = "../"  # where the outputs are saved

### (1) Identify training/test samples
posts = pd.read_csv(dir_data + "gendered_posts.csv")
keys_X = pd.read_csv(dir_data + "keys_to_X.csv")  # in the same order as rows in matrix X

# additional step to make sure the order is consistent with matrix "X" of word counts
# (This step may be unnecessary if you have sorted posts by title_id and post_id)
keys_merged = pd.merge(keys_X, posts, on=['title_id', 'post_id'], how="left")

# note: "non-duplicate" posts contain only female or only male classifiers
i_train = np.where(keys_merged['training'] == 1) # 75% of non-duplicate posts as training sample
i_test0 = np.where(keys_merged['training'] == 0) # 25% of non-duplicate posts as test sample for selecting optimal probability threshold
i_test1 = np.where(keys_merged['training'].isnull()) # duplicate posts that include both female and male classifiers; To be reclassified 

# an array of unambiguous gender in the training sample
# Updated: as_matrix() -> to_numpy()
y_train = keys_merged.loc[i_train[0], 'female'].to_numpy(dtype=int).ravel()

### (2) Bring in word count matrix X
# Updated: encoding arg removed; use allow_pickle and unwrap if needed
word_counts = np.load(dir_data + "X_word_count.npz", allow_pickle=True)
X = word_counts["X"]

# Some old .npz files store X as a 0-d object array, so to unwrap it safely:
if isinstance(X, np.ndarray) and X.dtype == object:
    X = X.item()

#Sanity check for row alignment
assert X.shape[0] == len(keys_X), "Row mismatch: X rows must equal keys_to_X rows"

X_train = X[i_train[0], :]
X_test0 = X[i_test0[0], :]
X_test1 = X[i_test1[0], :]

### (3) Select Predictors: most frequent 10K excluding gender classifiers & additional last names
vocab10K = pd.read_csv(dir_data + "vocab10K.csv")
vocab10K['exclude'].sum()

exclude_vocab = vocab10K.loc[vocab10K['exclude'] == 1, :]
i_exclude = (exclude_vocab['index'] - 1).astype(int)  # convert to 0-indexed ints

i_columns=range(10000)
i_keep_columns=list(set(i_columns)-set(i_exclude)) 

# ADDED SUGGESTION: keep column order stable
i_keep_columns = sorted(i_keep_columns)

np.savetxt(dir_lasso+"i_keep_columns.txt",i_keep_columns) # later this can be merged by estimated coefficients (in the same order as these indices) 

X_train = X_train[:, i_keep_columns]
print(X_train.shape)
X_test0 = X_test0[:, i_keep_columns]
print(X_test0.shape)
X_test1 = X_test1[:, i_keep_columns]
print(X_test1.shape)

################################################################################################################
#                                     ### logistic LASSO Model ###
################################################################################################################
from sklearn.linear_model import LogisticRegressionCV
model=LogisticRegressionCV(Cs=20,cv=5,penalty='l1',solver='liblinear',refit=True, n_jobs=-1).fit(X_train,y_train)  # ADDED SUGGESTION: n_jobs=-1

coef=model.coef_
len(coef[0]) 
np.savetxt(dir_lasso+"coef_lasso_logit_full.txt",coef[0])


# predicted probability for a post being Female
ypred_train = model.predict_proba(X_train)[:, 1]  # Pr(female=1)
ypred_test0 = model.predict_proba(X_test0)[:, 1]
ypred_test1 = model.predict_proba(X_test1)[:, 1]

# ypred in the files below have been brought back into "gendered_posts.csv"
np.savetxt(dir_lasso + "ypred_train.txt", ypred_train)
np.savetxt(dir_lasso + "ypred_test0.txt", ypred_test0)
np.savetxt(dir_lasso + "ypred_test1.txt", ypred_test1)

#################################################################################################
#Linear Lasso Model with Pronoun Sample 
dir_data = "../"   # specify the directory to data files
dir_lasso = "../"  # where the outputs are saved

### (1) Identify training/test samples
posts = pd.read_csv(dir_data + "gendered_posts.csv")
keys_X = pd.read_csv(dir_data + "keys_to_X.csv")  # in the same order as rows in matrix X

# additional step to make sure the order is consistent with matrix "X" of word counts
# (This step may be unnecessary if you have sorted posts by title_id and post_id)
keys_merged = pd.merge(keys_X, posts, on=["title_id", "post_id"], how="left")

# note: "non-duplicate" posts contain only female or only male classifiers
i_train = np.where(keys_merged["training_pronoun"] == 1)  # 75% of non-duplicate posts as training sample
i_test0 = np.where(keys_merged["training_pronoun"] == 0)  # 25% as test sample for selecting optimal threshold
i_test1 = np.where((keys_merged["fem_pronoun"] > 0) & (keys_merged["male_pronoun"] > 0))  # duplicate posts that include both female and male classifiers; To be reclassified 

# an array of unambiguous gender in the training sample
# Updated: as_matrix() -> to_numpy()
y_train = keys_merged.loc[i_train[0], "female_pronoun"].to_numpy(dtype=int).ravel()

### (2) Bring in word count matrix X
# Updated: encoding arg removed; use allow_pickle to unwrap
word_counts = np.load(dir_data + "X_word_count.npz", allow_pickle=True)
X = word_counts["X"]

# X is stored as a 0-d object array, so unwrap it
if isinstance(X, np.ndarray) and X.dtype == object:
    X = X.item()

# ADDED SUGGESTION: sanity check row alignment
assert X.shape[0] == len(keys_X), "Row mismatch: X rows must equal keys_to_X rows"

X_train = X[i_train[0], :]
X_test0 = X[i_test0[0], :]
X_test1 = X[i_test1[0], :]

### (3) Select Predictors: most frequent 10K excluding gender classifiers & additional last names
vocab10K = pd.read_csv(dir_data + "vocab10K.csv")
vocab10K["exclude"].sum()

exclude_vocab = vocab10K.loc[vocab10K["exclude"] == 1, :]
i_exclude = (exclude_vocab["index"] - 1).astype(int) # indexing in Python starts from 0, while the indices for vocab are 1 to 10,000

i_columns = range(10000)

i_keep_columns = list(set(i_columns) - set(i_exclude))

# ADDED SUGGESTION: keep column order stable
i_keep_columns = sorted(i_keep_columns)

np.savetxt(dir_lasso + "i_keep_columns.txt", i_keep_columns, fmt="%d") # later this can be merged by estimated coefficients (in the same order as these indices) 

X_train = X_train[:, i_keep_columns]
print(X_train.shape)
X_test0 = X_test0[:, i_keep_columns]
print(X_test0.shape)
X_test1 = X_test1[:, i_keep_columns]
print(X_test1.shape)

################################################################################################################
#                                     ### Linear LASSO Model Pronoun Sample ###
################################################################################################################

from sklearn.linear_model import LassoCV
model=LassoCV(cv=5, n_jobs=-1).fit(X_train,y_train)  # ADDED SUGGESTION: n_jobs=-1

coef=model.coef_
type(coef)
len(coef)
np.savetxt(dir_lasso+"coef_lasso_linear_pronoun.txt",coef) 

yhat_train = model.predict(X_train)
yhat_test0 = model.predict(X_test0)
yhat_test1 = model.predict(X_test1)

np.savetxt("yhat_linear_train.txt", yhat_train)
np.savetxt("yhat_linear_test0.txt", yhat_test0)
np.savetxt("yhat_linear_test1.txt", yhat_test1)

################################################################################################################
#Logit Lasso Pronoun Sample 
import numpy as np
import pandas as pd

dir_data = "../"   # specify the directory to data files
dir_lasso = "../"  # where the outputs are saved

### (1) Identify training/test samples
posts = pd.read_csv(dir_data + "gendered_posts.csv")
keys_X = pd.read_csv(dir_data + "keys_to_X.csv")  # in the same order as rows in matrix X

# additional step to make sure the order is consistent with the matrix "X" of word counts
# (This step may be unnecessary if you have sorted posts by title_id and post_id)
keys_merged = pd.merge(keys_X, posts, on=["title_id", "post_id"], how="left")

# note: "non-duplicate" posts contain only female or only male classifiers
i_train = np.where(keys_merged["training_pronoun"] == 1)  # 75% of non-duplicate posts as training sample
i_test0 = np.where(keys_merged["training_pronoun"] == 0)  # 25% of non-duplicate posts as test sample for selecting optimal 
i_test1 = np.where((keys_merged["fem_pronoun"] > 0) & (keys_merged["male_pronoun"] > 0))  # duplicate posts that include both female and male classifiers; To be reclassified 

# an array of unambiguous gender in the training sample
# Updated: as_matrix() -> to_numpy()
y_train = keys_merged.loc[i_train[0], "female_pronoun"].to_numpy(dtype=int).ravel()

### (2) Bring in word count matrix X
# Updated: encoding removed; allow_pickle True to unwrap
word_counts = np.load(dir_data + "X_word_count.npz", allow_pickle=True)
X = word_counts["X"]

# Unwrap since stored as 0-d object array
if isinstance(X, np.ndarray) and X.dtype == object:
    X = X.item()

# ADDED SUGGESTION: sanity check row alignment
assert X.shape[0] == len(keys_X), "Row mismatch: X rows must equal keys_to_X rows"

X_train = X[i_train[0], :]
X_test0 = X[i_test0[0], :]
X_test1 = X[i_test1[0], :]

### (3) Select Predictors: most frequent 10K excluding gender classifiers & additional last names
vocab10K = pd.read_csv(dir_data + "vocab10K.csv")
vocab10K["exclude"].sum()

exclude_vocab = vocab10K.loc[vocab10K["exclude"] == 1, :]
i_exclude = (exclude_vocab["index"] - 1).astype(int) # indexing in Python starts from 0, while the indices for vocab are 1 to 10,000

i_columns = range(10000)

i_keep_columns = list(set(i_columns) - set(i_exclude))

# ADDED SUGGESTION: keep column order stable
i_keep_columns = sorted(i_keep_columns)

np.savetxt(dir_lasso + "i_keep_columns.txt", i_keep_columns, fmt="%d")

X_train = X_train[:, i_keep_columns]
print(X_train.shape)
X_test0 = X_test0[:, i_keep_columns]
print(X_test0.shape)
X_test1 = X_test1[:, i_keep_columns]
print(X_test1.shape)

################################################################################################################
#                                    ### logistic LASSO Model Pronoun Sample ###
################################################################################################################

from sklearn.linear_model import LogisticRegressionCV
model=LogisticRegressionCV(Cs=20,cv=5,penalty='l1',solver='liblinear',refit=True, n_jobs=-1).fit(X_train,y_train)  # ADDED SUGGESTION: n_jobs=-1

coef=model.coef_
len(coef[0]) 
np.savetxt(dir_lasso+"coef_lasso_logit_pronoun.txt",coef[0])

# predicted probability for a post being Female
ypred_train = model.predict_proba(X_train)[:, 1]  # Pr(female=1)
ypred_test0 = model.predict_proba(X_test0)[:, 1]
ypred_test1 = model.predict_proba(X_test1)[:, 1]

# "ypred_pronoun" in the files below have been brought back into "gendered_posts.csv"
np.savetxt(dir_lasso + "ypred_pronoun_train.txt", ypred_train)
np.savetxt(dir_lasso + "ypred_pronoun_test0.txt", ypred_test0)
np.savetxt(dir_lasso + "ypred_pronoun_test1.txt", ypred_test1)
#################################################################################################################################


(300788, 9540)
(99941, 9540)
(44081, 9540)
